# NEU Dataset EDA

This notebook summarizes dataset quality, class distribution, split counts, image size distribution, and sample images.

In [ ]:
from pathlib import Path
import json
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
ROOT = Path('..')
SPLITS_DIR = ROOT / 'data' / 'splits'
METRICS_DIR = ROOT / 'metrics'

if not SPLITS_DIR.exists():
    raise FileNotFoundError('Run scripts/create_splits.py first.')

In [ ]:
def gather_files(split):
    split_dir = SPLITS_DIR / split
    items = []
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        for img_path in class_dir.rglob('*'):
            if img_path.is_file():
                items.append((img_path, class_dir.name, split))
    return items

records = []
for split_name in ['train', 'val', 'test']:
    records.extend(gather_files(split_name))

print(f'Total images: {len(records)}')

In [ ]:
class_counts = Counter([label for _, label, _ in records])
split_counts = Counter([split for _, _, split in records])

print('Class counts:')
for label, count in sorted(class_counts.items()):
    print(label, count)

print('\nSplit counts:')
for split, count in split_counts.items():
    print(split, count)

In [ ]:
plt.figure(figsize=(8, 4))
labels = sorted(class_counts.keys())
values = [class_counts[k] for k in labels]
plt.bar(labels, values)
plt.xticks(rotation=30, ha='right')
plt.title('Samples per Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
sizes = []
for path, _, _ in records:
    with Image.open(path) as img:
        sizes.append(img.size)

widths = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

plt.figure(figsize=(6, 4))
plt.scatter(widths, heights, alpha=0.3)
plt.title('Image Size Distribution')
plt.xlabel('Width')
plt.ylabel('Height')
plt.tight_layout()
plt.show()

In [ ]:
random.seed(42)
sample_items = random.sample(records, min(9, len(records)))

plt.figure(figsize=(9, 9))
for i, (path, label, split) in enumerate(sample_items, 1):
    with Image.open(path) as img:
        arr = np.array(img)
    plt.subplot(3, 3, i)
    if arr.ndim == 2:
        plt.imshow(arr, cmap='gray')
    else:
        plt.imshow(arr)
    plt.title(f'{label} ({split})', fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Observations Template

- Class imbalance notes: fill after plotting class counts.
- Data quality notes: invalid/duplicate removal from cleaning summary.
- Recommended next step: targeted augmentation for weak classes.